# Machine Learning Basics

## 🎯 Learning Objectives
By the end of this notebook, you will understand:
- What is Machine Learning?
- Supervised vs Unsupervised learning
- Train-test split and cross-validation
- Feature scaling and normalization
- Model evaluation metrics
- Scikit-learn workflow

**Estimated Time**: 90 minutes

---

## 1. What is Machine Learning?

### Traditional Programming vs Machine Learning

**Traditional Programming:**
```
Rules + Data → Output
```
You write explicit rules:
```python
if temperature > 30:
    turn_on_ac()
```

**Machine Learning:**
```
Data + Output → Rules (Model)
```
The algorithm learns patterns from data:
- Show it 1000s of examples
- It finds patterns automatically
- Makes predictions on new data

### Types of Machine Learning

1. **Supervised Learning**: Learn from labeled data
   - Regression: Predict numbers (e.g., energy consumption)
   - Classification: Predict categories (e.g., normal vs anomaly)

2. **Unsupervised Learning**: Find patterns in unlabeled data
   - Clustering: Group similar items
   - Anomaly Detection: Find unusual patterns

3. **Reinforcement Learning**: Learn through trial and error (not covered here)

## 2. Scikit-learn - The ML Library

Scikit-learn is Python's main ML library. It provides:
- Dozens of algorithms
- Simple, consistent API
- Built-in datasets
- Model evaluation tools

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import datasets

print("✅ Libraries imported successfully!")

## 3. Supervised Learning - Regression Example

### Problem: Predict Energy Consumption from Temperature

In [ ]:
# Create synthetic energy data
np.random.seed(42)
n_samples = 200

# Temperature (°C)
temperature = np.random.uniform(15, 35, n_samples)

# Energy consumption (kWh) - increases with temperature (AC usage)
# Base consumption + temperature effect + noise
energy = 1000 + (temperature - 20) * 50 + np.random.normal(0, 100, n_samples)

# Create DataFrame
data = pd.DataFrame({
    'temperature': temperature,
    'energy_kwh': energy
})

print(data.head())
print(f"\nDataset size: {len(data)} samples")

# Visualize the relationship
plt.figure(figsize=(10, 6))
plt.scatter(data['temperature'], data['energy_kwh'], alpha=0.6, s=30)
plt.xlabel('Temperature (°C)')
plt.ylabel('Energy Consumption (kWh)')
plt.title('Temperature vs Energy Consumption')
plt.grid(True, alpha=0.3)
plt.show()

print(f"\nCorrelation: {data['temperature'].corr(data['energy_kwh']):.3f}")

### Train-Test Split - The Golden Rule

**Never test on data you trained on!**

- **Training Set (80%)**: Learn patterns
- **Test Set (20%)**: Evaluate performance

In [ ]:
from sklearn.model_selection import train_test_split

# Prepare data
X = data[['temperature']]  # Features (must be 2D)
y = data['energy_kwh']  # Target

# Split into train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training set size:", len(X_train))
print("Test set size:", len(X_test))
print("\nX_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

### Training a Model - Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Create model
model = LinearRegression()

# Train (fit) the model
model.fit(X_train, y_train)

print("✅ Model trained!")
print(f"\nCoefficient (slope): {model.coef_[0]:.2f}")
print(f"Intercept: {model.intercept_:.2f}")
print(f"\nFormula: Energy = {model.intercept_:.2f} + {model.coef_[0]:.2f} * Temperature")

### Making Predictions

In [ ]:
# Predict on test set
y_pred = model.predict(X_test)

# Show some predictions
comparison = pd.DataFrame({
    'Temperature': X_test['temperature'].values[:10],
    'Actual': y_test.values[:10],
    'Predicted': y_pred[:10],
    'Error': np.abs(y_test.values[:10] - y_pred[:10])
})

print("First 10 predictions:")
print(comparison.to_string(index=False))

### Evaluating the Model

In [ ]:
# Calculate metrics
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("Model Performance:")
print(f"  Mean Squared Error: {mse:.2f}")
print(f"  Root Mean Squared Error: {rmse:.2f} kWh")
print(f"  R² Score: {r2:.3f} (closer to 1.0 is better)")

# Visualize predictions
plt.figure(figsize=(12, 5))

# Subplot 1: Predicted vs Actual
plt.subplot(1, 2, 1)
plt.scatter(y_test, y_pred, alpha=0.6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
         'r--', linewidth=2, label='Perfect predictions')
plt.xlabel('Actual Energy (kWh)')
plt.ylabel('Predicted Energy (kWh)')
plt.title('Predicted vs Actual')
plt.legend()
plt.grid(True, alpha=0.3)

# Subplot 2: Regression line
plt.subplot(1, 2, 2)
plt.scatter(X_test, y_test, alpha=0.6, label='Actual')
plt.plot(X_test, y_pred, 'r-', linewidth=2, label='Model prediction')
plt.xlabel('Temperature (°C)')
plt.ylabel('Energy (kWh)')
plt.title('Model Fit')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Classification Example - Normal vs Anomaly

Let's classify energy readings as normal or anomaly.

In [ ]:
# Create classification dataset
np.random.seed(42)
n_normal = 180
n_anomaly = 20

# Normal readings
normal_energy = np.random.normal(1500, 150, n_normal)
normal_temp = np.random.normal(22, 2, n_normal)
normal_labels = np.zeros(n_normal)  # 0 = normal

# Anomalous readings (high energy, varied temp)
anomaly_energy = np.random.normal(2500, 200, n_anomaly)
anomaly_temp = np.random.uniform(15, 30, n_anomaly)
anomaly_labels = np.ones(n_anomaly)  # 1 = anomaly

# Combine
energy_all = np.concatenate([normal_energy, anomaly_energy])
temp_all = np.concatenate([normal_temp, anomaly_temp])
labels = np.concatenate([normal_labels, anomaly_labels])

# Create DataFrame
clf_data = pd.DataFrame({
    'energy_kwh': energy_all,
    'temperature': temp_all,
    'is_anomaly': labels
})

print(clf_data.head(10))
print(f"\nNormal samples: {(labels == 0).sum()}")
print(f"Anomaly samples: {(labels == 1).sum()}")

In [ ]:
# Visualize
plt.figure(figsize=(10, 6))
colors = ['green' if label == 0 else 'red' for label in labels]
plt.scatter(temp_all, energy_all, c=colors, alpha=0.6, s=50)
plt.xlabel('Temperature (°C)')
plt.ylabel('Energy (kWh)')
plt.title('Normal (green) vs Anomaly (red)')
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='green', label='Normal'),
                  Patch(facecolor='red', label='Anomaly')]
plt.legend(handles=legend_elements)
plt.grid(True, alpha=0.3)
plt.show()

### Train a Classifier - Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Prepare data
X_clf = clf_data[['energy_kwh', 'temperature']]
y_clf = clf_data['is_anomaly']

# Split
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

# Train classifier
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train_clf, y_train_clf)

# Predict
y_pred_clf = clf.predict(X_test_clf)

# Evaluate
accuracy = accuracy_score(y_test_clf, y_pred_clf)
print(f"Accuracy: {accuracy:.2%}")
print("\nClassification Report:")
print(classification_report(y_test_clf, y_pred_clf, target_names=['Normal', 'Anomaly']))

### Confusion Matrix - Understanding Errors

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test_clf, y_pred_clf)

# Visualize
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Normal', 'Anomaly'],
            yticklabels=['Normal', 'Anomaly'])
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix')
plt.show()

print("Confusion Matrix Explanation:")
print(f"  True Negatives (Normal predicted as Normal): {cm[0,0]}")
print(f"  False Positives (Normal predicted as Anomaly): {cm[0,1]}")
print(f"  False Negatives (Anomaly predicted as Normal): {cm[1,0]}")
print(f"  True Positives (Anomaly predicted as Anomaly): {cm[1,1]}")

## 5. Feature Scaling - Critical for ML!

Many algorithms perform better when features are on similar scales.

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Sample data with different scales
sample_data = pd.DataFrame({
    'energy_kwh': [1500, 1600, 1400],
    'temperature': [22, 25, 20],
    'building_area': [5000, 8000, 3000]  # Much larger scale!
})

print("Original data:")
print(sample_data)

# StandardScaler: mean=0, std=1
standard_scaler = StandardScaler()
scaled_standard = standard_scaler.fit_transform(sample_data)

print("\nStandardized (mean=0, std=1):")
print(pd.DataFrame(scaled_standard, columns=sample_data.columns))

# MinMaxScaler: scale to [0, 1]
minmax_scaler = MinMaxScaler()
scaled_minmax = minmax_scaler.fit_transform(sample_data)

print("\nMin-Max Scaled [0, 1]:")
print(pd.DataFrame(scaled_minmax, columns=sample_data.columns))

## 6. Cross-Validation - Better Evaluation

Single train-test split might be lucky/unlucky. Cross-validation uses multiple splits.

In [ ]:
from sklearn.model_selection import cross_val_score

# Using our earlier regression data
X_full = data[['temperature']]
y_full = data['energy_kwh']

# 5-fold cross-validation
model = LinearRegression()
cv_scores = cross_val_score(model, X_full, y_full, cv=5, scoring='r2')

print("Cross-Validation R² Scores:")
for i, score in enumerate(cv_scores, 1):
    print(f"  Fold {i}: {score:.3f}")

print(f"\nMean R² Score: {cv_scores.mean():.3f} (±{cv_scores.std():.3f})")

# Visualize
plt.figure(figsize=(8, 5))
plt.bar(range(1, 6), cv_scores, color='steelblue', alpha=0.7)
plt.axhline(y=cv_scores.mean(), color='red', linestyle='--', label=f'Mean: {cv_scores.mean():.3f}')
plt.xlabel('Fold')
plt.ylabel('R² Score')
plt.title('Cross-Validation Scores')
plt.legend()
plt.ylim([0, 1])
plt.grid(True, alpha=0.3)
plt.show()

## 7. The ML Workflow - Step by Step

Here's the standard workflow for any ML project:

In [ ]:
# STEP 1: Load and explore data
# (We've already done this)

# STEP 2: Prepare features and target
X = data[['temperature']]
y = data['energy_kwh']

# STEP 3: Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# STEP 4: Scale features (if needed)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # Use same scaler!

# STEP 5: Train model
model = LinearRegression()
model.fit(X_train_scaled, y_train)

# STEP 6: Make predictions
y_pred = model.predict(X_test_scaled)

# STEP 7: Evaluate
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Final Model Performance:")
print(f"  RMSE: {rmse:.2f} kWh")
print(f"  R² Score: {r2:.3f}")

# STEP 8: Use for new predictions
new_temperature = np.array([[28.5]])  # 28.5°C
new_temp_scaled = scaler.transform(new_temperature)
predicted_energy = model.predict(new_temp_scaled)

print(f"\nPrediction: At {new_temperature[0,0]}°C, expected energy: {predicted_energy[0]:.0f} kWh")

## 🎉 Congratulations!

You've learned ML fundamentals! You can now:

✅ Understand supervised vs unsupervised learning  
✅ Split data into train and test sets  
✅ Train regression and classification models  
✅ Evaluate models with appropriate metrics  
✅ Scale features correctly  
✅ Use cross-validation  
✅ Follow the complete ML workflow  

### Next Steps:
Move on to **04_Time_Series_Analysis.ipynb** to learn about temporal patterns!

---

### 📚 Additional Resources:
- [Scikit-learn User Guide](https://scikit-learn.org/stable/user_guide.html)
- [Google's ML Crash Course](https://developers.google.com/machine-learning/crash-course)
- [Andrew Ng's ML Course](https://www.coursera.org/learn/machine-learning)